In [1]:
# add .. path 
import os
import sys
sys.path.append('..')
import utils.llm_training as llm_training
import utils.llm_configs as llm_configs

import logging

# --- Basic Configuration ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - [%(name)s] - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)

os.environ["WANDB_PROJECT"]="medex_fine_tuning"


In [2]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("medexanon/Medex")['train']

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/21 [00:00<?, ?it/s]

In [3]:
ds_subset = ds.select(range(10000))


In [4]:
# === Cell 1: Configuration ===
model_config = llm_configs.ModelConfig(
    id="Qwen/Qwen2.5-0.5B",
    peft=llm_configs.PeftConfig(
        enabled=False,
        add_eot_token=False,  # No longer doing EOT token for LIMA
    ),
    quantization=llm_configs.QuantizationConfig(mode=None), # Use QLoRA
)

log.info("--- Configuration ---")
print(model_config.model_dump_json(indent=2))

log.info("\n--- Loading Model for Training ---")
model, tokenizer = llm_training.load_model_for_training(model_config, log)

2025-07-08 16:00:49 - INFO - [__main__] - --- Configuration ---
2025-07-08 16:00:49 - INFO - [__main__] - 
--- Loading Model for Training ---
2025-07-08 16:00:49 - INFO - [__main__] - Loading model 'Qwen/Qwen2.5-0.5B' for training...


{
  "id": "Qwen/Qwen2.5-0.5B",
  "torch_dtype": "auto",
  "attn_implementation": "flash_attention_2",
  "peft": {
    "enabled": false,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "add_eot_token": false
  },
  "quantization": {
    "mode": null
  }
}


2025-07-08 16:00:51 - INFO - [__main__] - Model and tokenizer loaded successfully.


In [5]:
def concat_columns(example, tokenizer):
    """
    Combine DOI/entity/fact/MolInfo/GeneInfo into one human-readable string.
    Empty or missing fields are omitted for that row.
    """

    chunks = []

    # 1) flat string columns
    if example.get("DOI"):
        chunks.append(f"[DOI] {example['DOI']}")
    if example.get("entity"):
        chunks.append(f"[entity] {example['entity']}")
    if example.get("fact"):
        chunks.append(f"[fact] {example['fact']}")

    # 2) MolInfo → [SMILES] …
    mol = example.get("MolInfo")
    if isinstance(mol, dict):
        smiles = mol.get("SMILES")
        if smiles:
            chunks.append(f'[SMILES] "{smiles}"')

    # 3) GeneInfo → [GeneInfo] key: value, …
    gene = example.get("GeneInfo")
    if isinstance(gene, dict) and gene:
        def _fmt(key, val):
            return f'"{key}": {val}' if isinstance(val, int) else f'"{key}": "{val}"'
        fields = [_fmt(k, v) for k, v in gene.items() if v not in (None, "", [])]
        if fields:
            chunks.append(f"[GeneInfo] " + ", ".join(fields))

    # join all parts with a single space
    return {"text": " ".join(chunks) + tokenizer.eos_token}

# ---- apply to your Dataset ----
# creates a new 'text' column, keeps the originals (remove_columns=[] by default)
ds_with_text = ds_subset.map(concat_columns, fn_kwargs={"tokenizer": tokenizer},  desc="Building concatenated text")

In [6]:
medex_ds = ds_with_text.select_columns(["text"])
medex_ds

Dataset({
    features: ['text'],
    num_rows: 10000
})

In [7]:
lima_training_config = llm_configs.TrainingConfig(
    run_name = "2 million samples on medex",
    num_train_epochs = 1,
    learning_rate  = 4e-5,
    logging_strategy = "steps", 
    logging_steps = 10000,
    gradient_checkpointing=False,
    context_length = 4096,
    use_liger_kernel=True,
    per_device_batch_size = 4096,
    gradient_accumulation_steps=1,
    # warmup_steps  = 0, # LIMA specifies no warmup, so we set this explicitly
    warmup_ratio = 0.3, # Use our default warmup ratio instead
    packing=True,
    padding_free = True,
    sequential_sampling = False,
    reverse_ffd_packing= False,
    remove_unused_columns=False,
)


# === Run LIMA Fine-Tuning ===
log.info("\n--- Starting LIMA Fine-Tuning ---")
# The model object will be updated with the fine-tuned weights
trainer = llm_training.sft_train_on_dataset(
    model=model,
    tokenizer=tokenizer,
    log=log,
    train_dataset=medex_ds,
    train_cfg=lima_training_config,
    train=False,
    use_liger_loss = True
)

2025-07-08 16:00:52 - INFO - [__main__] - 
--- Starting LIMA Fine-Tuning ---
2025-07-08 16:00:52 - INFO - [__main__] - Starting SFT training run...


False


2025-07-08 16:00:52 - INFO - [liger_kernel.transformers.monkey_patch] - Applying Liger kernels to model instance with model type: qwen2 with kwargs: {}


Applied Liger kernels to Qwen2


In [8]:
print(len(trainer.get_train_dataloader()))

268


In [9]:
import wandb
trainer.train()
wandb.finish()

wandb: Currently logged in as: jiosephlee (upenn-ml) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x753bf61a5850>> (for post_run_cell), with arguments args (<ExecutionResult object at 753bde267f90, execution_count=9 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 753bde267d90, raw_cell="import wandb
trainer.train()
wandb.finish()" store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Brunpod/root/fine-tuning-or-retrieval/scripts/fine_tuning_on_medex.ipynb#X20sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe